# EC. Primer Parcial — Proyecto Práctico HPC
## Integrante 4 — Análisis y gráficas de rendimiento

**Rama:** `feature/graficas`

---

## Objetivo de esta parte

Este es el **cuarto notebook del proyecto**.

Los notebooks anteriores se encargaron de:

1. `01_secuencial.ipynb` — implementación secuencial.
2. `02_paralelo.ipynb` — implementación paralela.
3. `03_experimentos.ipynb` — promedios, Speedup y eficiencia.
4. `04_graficas.ipynb` — gráficas e interpretación de los resultados.

En esta etapa se utilizan los resultados ya calculados para generar gráficas mediante Python.

Las gráficas solicitadas son:

- workers vs. tiempo de ejecución;
- workers vs. Speedup;
- workers vs. eficiencia.

> Las gráficas se generan mediante código y se guardan en la carpeta `resultados/`.


# 1. Importar las librerías

Utilizaremos:

- `pandas` para leer los archivos CSV;
- `matplotlib` para generar las gráficas;
- `pathlib` para localizar correctamente la carpeta de resultados.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()

if (cwd / "resultados").exists():
    RESULTADOS = cwd / "resultados"
elif (cwd.parent / "resultados").exists():
    RESULTADOS = cwd.parent / "resultados"
else:
    raise FileNotFoundError("No se encontró la carpeta resultados/")

print("Carpeta de resultados:", RESULTADOS.resolve())


# 2. Cargar los resultados

El notebook de experimentación generó:

```text
resultados/analisis.csv
```

Este archivo contiene para cada número de workers:

- tiempo promedio;
- Speedup;
- eficiencia;
- comparación adicional contra la versión secuencial.

También cargaremos `secuencial.csv` para mostrar la línea de referencia secuencial en la gráfica de tiempos.


In [ ]:
analisis = pd.read_csv(RESULTADOS / "analisis.csv")
secuencial = pd.read_csv(RESULTADOS / "secuencial.csv")

display(analisis)


# 3. Obtener el tiempo secuencial promedio

La versión secuencial realizó tres pruebas.

Calcularemos nuevamente el promedio a partir de las tres mediciones para utilizarlo como referencia visual.


In [ ]:
secuencial_pruebas = secuencial[
    secuencial["prueba"].astype(str).str.lower() != "promedio"
].copy()

secuencial_pruebas["tiempo_segundos"] = pd.to_numeric(
    secuencial_pruebas["tiempo_segundos"]
)

tiempo_secuencial = secuencial_pruebas["tiempo_segundos"].mean()

print(f"Tiempo promedio secuencial: {tiempo_secuencial:.6f} s")


# 4. Gráfica: workers vs. tiempo de ejecución

Esta es la gráfica principal solicitada en la práctica.

En el eje X se muestra el número de workers.

En el eje Y se muestra el tiempo promedio en segundos.

También se agrega una línea horizontal con el tiempo de la versión secuencial para facilitar la comparación.


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    analisis["workers"],
    analisis["tiempo_promedio_segundos"],
    marker="o",
    label="Paralelo"
)

plt.axhline(
    tiempo_secuencial,
    linestyle="--",
    label="Secuencial"
)

plt.xlabel("Número de workers")
plt.ylabel("Tiempo promedio (segundos)")
plt.title("Workers vs. tiempo de ejecución")
plt.xticks(analisis["workers"])
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

ruta = RESULTADOS / "tiempo_workers.png"
plt.savefig(ruta, dpi=150)
plt.show()

print("Gráfica guardada en:", ruta)


## Interpretación

Un tiempo menor significa mejor rendimiento.

Con los datos actuales, el tiempo de la versión paralela disminuye al pasar de 1 a 2 y 4 workers.

Sin embargo, la línea de referencia permite observar si la implementación paralela supera o no a la versión secuencial.


# 5. Gráfica: workers vs. Speedup

El Speedup indica cuánto se acelera la ejecución respecto a la configuración con 1 worker.

\[
S_p = \frac{T_1}{T_p}
\]

Un Speedup de 2 significa que la configuración es aproximadamente dos veces más rápida que la referencia con 1 worker.


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    analisis["workers"],
    analisis["speedup"],
    marker="o"
)

plt.xlabel("Número de workers")
plt.ylabel("Speedup")
plt.title("Workers vs. Speedup")
plt.xticks(analisis["workers"])
plt.grid(alpha=0.25)
plt.tight_layout()

ruta = RESULTADOS / "speedup.png"
plt.savefig(ruta, dpi=150)
plt.show()

print("Gráfica guardada en:", ruta)


# 6. Gráfica: workers vs. eficiencia

La eficiencia indica qué tan bien se aprovechan los workers disponibles.

\[
E_p = \frac{S_p}{p}
\]

Una eficiencia de `1.0` representa un aprovechamiento ideal.

En la práctica suele disminuir al agregar workers debido al costo de crear procesos, repartir datos y combinar resultados.


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    analisis["workers"],
    analisis["eficiencia"],
    marker="o"
)

plt.xlabel("Número de workers")
plt.ylabel("Eficiencia")
plt.title("Workers vs. eficiencia")
plt.xticks(analisis["workers"])
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.tight_layout()

ruta = RESULTADOS / "eficiencia.png"
plt.savefig(ruta, dpi=150)
plt.show()

print("Gráfica guardada en:", ruta)


# 7. Tabla final de resultados

Antes de redactar las conclusiones, mostramos las métricas principales en una sola tabla.


In [ ]:
tabla_final = analisis[
    [
        "workers",
        "tiempo_promedio_segundos",
        "speedup",
        "eficiencia",
        "speedup_vs_secuencial",
    ]
].copy()

tabla_final["tiempo_promedio_segundos"] = (
    tabla_final["tiempo_promedio_segundos"].round(6)
)
tabla_final["speedup"] = tabla_final["speedup"].round(4)
tabla_final["eficiencia"] = tabla_final["eficiencia"].round(4)
tabla_final["speedup_vs_secuencial"] = (
    tabla_final["speedup_vs_secuencial"].round(4)
)

display(tabla_final)


# 8. Análisis de los resultados

## ¿La ejecución paralela fue más rápida que la secuencial?

Con las mediciones actuales, **no**.

La versión secuencial obtuvo un tiempo promedio menor que las configuraciones paralelas.

Esto no significa que el experimento haya fallado. La práctica indica que un peor tiempo al utilizar más workers sigue siendo un resultado válido siempre que esté correctamente medido y explicado.

## ¿Qué número de workers obtuvo el menor tiempo?

Dentro de las configuraciones paralelas, **4 workers** obtuvo el menor tiempo promedio.

## ¿Duplicar el número de workers duplicó el rendimiento?

No.

El Speedup aumenta, pero no de manera perfectamente proporcional al número de workers.

Esto ocurre porque la paralelización introduce costos adicionales como:

- creación de procesos;
- división de los datos;
- transferencia de información;
- combinación de resultados;
- competencia por recursos del procesador.

## ¿Por qué el problema puede paralelizarse?

Cada fila puede procesarse de forma independiente.

El cálculo de una fila no necesita el resultado de otra fila.

Por eso es posible dividir el dataset en bloques y asignar cada bloque a un worker diferente.

## ¿En qué momento agregar más workers deja de ser beneficioso?

Agregar workers deja de ser beneficioso cuando el costo de administrar los procesos adicionales es comparable o superior a la mejora obtenida.

También influye la cantidad de núcleos físicos y lógicos disponibles.

## ¿Este experimento representa HPC?

El experimento demuestra **principios utilizados en HPC**.

No se está utilizando un clúster o supercomputador, pero sí se estudian conceptos fundamentales como paralelización, Speedup, eficiencia, escalabilidad y medición del rendimiento.


# 9. Archivos generados

Después de ejecutar este notebook deben existir:

```text
resultados/
├── secuencial.csv
├── paralelo.csv
├── analisis.csv
├── tiempo_workers.png
├── speedup.png
└── eficiencia.png
```

Estos archivos forman parte de la evidencia experimental del proyecto.


# 10. Conclusión

Los resultados permiten observar que usar más workers puede reducir el tiempo dentro de la implementación paralela, pero el rendimiento no aumenta de manera proporcional.

En este experimento, 4 workers fue la mejor configuración paralela, aunque la versión secuencial siguió obteniendo un menor tiempo.

Esto demuestra una idea importante de HPC:

> **Paralelizar un problema tiene un costo y utilizar más workers no garantiza automáticamente que un programa sea más rápido.**
